In [23]:
import csv
import datetime
import cv2
import os

In [2]:
def time_to_seconds(time_str):
    """Convert a time string (MM:SS.s or HH:MM:SS.s) into seconds."""
    parts = time_str.split(':')
    if len(parts) == 2:
        minutes = float(parts[0])
        seconds = float(parts[1])
        return minutes * 60 + seconds
    elif len(parts) == 3:
        hours = float(parts[0])
        minutes = float(parts[1])
        seconds = float(parts[2])
        return hours * 3600 + minutes * 60 + seconds
    else:
        raise ValueError("Unexpected time format: " + time_str)

In [16]:
def add_seconds_to_time_str(time_str, seconds_to_add):
    """
    Given a time string in the format HH:MM:SS.microseconds (or MM:SS.microseconds),
    add a specified number of seconds and return the updated time string.
    """
    # Try HH:MM:SS.microseconds first
    try:
        dt = datetime.datetime.strptime(time_str, "%H:%M:%S.%f")
    except ValueError:
        # Otherwise try MM:SS.microseconds
        dt = datetime.datetime.strptime(time_str, "%M:%S.%f")
    new_dt = dt + datetime.timedelta(seconds=seconds_to_add)
    # Return in same format (always with microseconds)
    if new_dt.hour > 0:
        return new_dt.strftime("%H:%M:%S.%f")
    else:
        return new_dt.strftime("%M:%S.%f")

In [21]:
input_csv = "/Volumes/TVault2/datasets/sugvu24/labels/case_000/tools.csv"       # Replace with your CSV path
video_file = "/Volumes/TVault2/datasets/sugvu24/case_000/case_000_video_part_001.mp4"

In [27]:
frames_output_dir = "/Volumes/TVault2/datasets/sugvu24/extracted_frames"
os.makedirs(output_dir, exist_ok=True)
output_csv = "/Volumes/TVault2/datasets/sugvu24/extracted_frames/" + "output.csv"

In [19]:
cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Video FPS: {fps}")

Video FPS: 60.0


In [31]:
from datetime import timedelta, datetime
import os
import csv
import cv2
import numpy as np

def parse_time_str(time_str):
    formats = ["%H:%M:%S.%f", "%M:%S.%f", "%H:%M:%S"]
    for fmt in formats:
        try:
            dt = datetime.strptime(time_str, fmt)
            return timedelta(
                hours=dt.hour, minutes=dt.minute, seconds=dt.second, microseconds=dt.microsecond
            )
        except ValueError:
            continue
    raise ValueError(f"Time data '{time_str}' does not match any expected format.")

def add_seconds_to_time_str(time_str, seconds_to_add):
    td = parse_time_str(time_str)
    new_td = td + timedelta(seconds=seconds_to_add)
    total_seconds = int(new_td.total_seconds())
    microseconds = new_td.microseconds
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.{microseconds:06d}"

def time_to_seconds(time_str):
    td = parse_time_str(time_str)
    return td.total_seconds()

def is_frame_valid(frame):
    if frame is None or frame.size == 0:
        return False
    mean_value = np.mean(frame)
    return mean_value > 5

def extract_frame_at_time(cap, seconds, fps):
    frame_number = int(seconds * fps)

    # Try a sequence of approaches to get a valid frame

    # Try direct time seeking
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ret, frame = cap.read()
    if ret and is_frame_valid(frame):
        return True, frame

    # Try frame number seeking
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    if ret and is_frame_valid(frame):
        return True, frame

    # Try nearby frames
    for offset in [-30, -20, -10, 10, 20, 30]:
        nearby_frame = max(0, frame_number + offset)
        cap.set(cv2.CAP_PROP_POS_FRAMES, nearby_frame)
        ret, frame = cap.read()
        if ret and is_frame_valid(frame):
            return True, frame

    # Try nearby times
    for offset in [-3, -2, -1, 1, 2, 3]:
        nearby_time = max(0, seconds + offset)
        cap.set(cv2.CAP_PROP_POS_MSEC, nearby_time * 1000)
        ret, frame = cap.read()
        if ret and is_frame_valid(frame):
            return True, frame

    return False, None

os.makedirs(frames_output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_file)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {video_file}")
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
video_duration = total_frames / fps
print(f"Video FPS: {fps}")
print(f"Total frames: {total_frames}, Duration: {video_duration:.2f} seconds")

with open(input_csv, newline='') as csvfile_in, open(output_csv, 'w', newline='') as csvfile_out:
    reader = csv.DictReader(csvfile_in, delimiter=',')
    fieldnames = reader.fieldnames + ['adjusted_install_case_time', 'frame_filename']
    writer = csv.DictWriter(csvfile_out, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        try:
            original_time = row['install_case_time']
            adjusted_time_str = add_seconds_to_time_str(original_time, 4)
            adjusted_sec = time_to_seconds(adjusted_time_str)
        except Exception as e:
            print(f"Error processing row {row.get('index', 'unknown')}: {e}")
            row['frame_filename'] = ""
            row['adjusted_install_case_time'] = ""
            writer.writerow(row)
            continue

        if adjusted_sec > video_duration:
            print(f"Adjusted time {adjusted_sec} seconds exceeds video duration for row {row['index']}.")
            row['frame_filename'] = ""
            row['adjusted_install_case_time'] = adjusted_time_str
            writer.writerow(row)
            continue

        ret, frame = extract_frame_at_time(cap, adjusted_sec, fps)

        if not ret or not is_frame_valid(frame):
            print(f"Could not read valid frame for row {row['index']} at {adjusted_time_str}.")
            row['frame_filename'] = ""
        else:
            instrument = row['groundtruth_toolname'].strip() or row['commercial_toolname'].strip()
            instrument_clean = instrument.replace(" ", "_")
            frame_filename = f"row{row['index']}_{instrument_clean}.png"
            frame_filepath = os.path.join(frames_output_dir, frame_filename)

            success = cv2.imwrite(frame_filepath, frame)
            if not success:
                print(f"Failed to write frame to {frame_filepath}")
                row['frame_filename'] = ""
            else:
                file_size = os.path.getsize(frame_filepath)
                if file_size < 10000:
                    os.remove(frame_filepath)
                    print(f"Removed suspiciously small file: {frame_filename} ({file_size} bytes)")
                    row['frame_filename'] = ""
                else:
                    row['frame_filename'] = frame_filename
                    print(f"Saved frame for row {row['index']} at {adjusted_time_str} as {frame_filename}")

        row['adjusted_install_case_time'] = adjusted_time_str
        writer.writerow(row)

cap.release()
print(f"New CSV file written to {output_csv}")

Video FPS: 60.0
Total frames: 151133.0, Duration: 2518.88 seconds
Saved frame for row 0 at 00:07:28.796000 as row0_stapler.png
Saved frame for row 1 at 00:08:01.979000 as row1_cadiere_forceps.png
Saved frame for row 2 at 00:08:48.729000 as row2_cadiere_forceps.png
Saved frame for row 3 at 00:17:22.659000 as row3_vessel_sealer.png
Could not read valid frame for row 4 at 00:23:07.499000.
Could not read valid frame for row 5 at 00:23:07.499000.
Could not read valid frame for row 6 at 00:24:58.379000.
Could not read valid frame for row 7 at 00:24:58.379000.
Could not read valid frame for row 8 at 00:24:58.379000.
Could not read valid frame for row 9 at 00:25:00.579000.
Could not read valid frame for row 10 at 00:25:00.579000.
Could not read valid frame for row 11 at 00:25:05.359000.
Could not read valid frame for row 12 at 00:26:56.959000.
Could not read valid frame for row 13 at 00:26:56.959000.
Saved frame for row 14 at 00:27:59.049000 as row14_Unknown_Instrument.png
Saved frame for row 

In [29]:
def parse_time_str(time_str):
    """
    Parse a time string into a datetime.timedelta.
    Attempts several common formats.
    """
    from datetime import timedelta, datetime
    formats = ["%H:%M:%S.%f", "%M:%S.%f", "%H:%M:%S"]
    for fmt in formats:
        try:
            dt = datetime.strptime(time_str, fmt)
            return timedelta(
                hours=dt.hour, minutes=dt.minute, seconds=dt.second, microseconds=dt.microsecond
            )
        except ValueError:
            continue
    raise ValueError(f"Time data '{time_str}' does not match any expected format.")

def add_seconds_to_time_str(time_str, seconds_to_add):
    """
    Add a specified number of seconds to the given time string.
    """
    td = parse_time_str(time_str)
    new_td = td + datetime.timedelta(seconds=seconds_to_add)
    total_seconds = int(new_td.total_seconds())
    microseconds = new_td.microseconds
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.{microseconds:06d}"

def time_to_seconds(time_str):
    """
    Convert a time string (in HH:MM:SS.microseconds or similar format) into seconds.
    """
    td = parse_time_str(time_str)
    return td.total_seconds()


os.makedirs(frames_output_dir, exist_ok=True)

# Open the video file using OpenCV
cap = cv2.VideoCapture(video_file)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {video_file}")
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
video_duration = total_frames / fps
print(f"Video FPS: {fps}")
print(f"Total frames: {total_frames}, Duration: {video_duration:.2f} seconds")

with open(input_csv, newline='') as csvfile_in, open(output_csv, 'w', newline='') as csvfile_out:
    reader = csv.DictReader(csvfile_in, delimiter=',')
    fieldnames = reader.fieldnames + ['adjusted_install_case_time', 'frame_filename']
    writer = csv.DictWriter(csvfile_out, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        try:
            original_time = row['install_case_time']
            adjusted_time_str = add_seconds_to_time_str(original_time, 4)
            adjusted_sec = time_to_seconds(adjusted_time_str)
        except Exception as e:
            print(f"Error processing row {row.get('index', 'unknown')}: {e}")
            row['frame_filename'] = ""
            row['adjusted_install_case_time'] = ""
            writer.writerow(row)
            continue

        if adjusted_sec > video_duration:
            print(f"Adjusted time {adjusted_sec} seconds exceeds video duration for row {row['index']}.")
            row['frame_filename'] = ""
            row['adjusted_install_case_time'] = adjusted_time_str
            writer.writerow(row)
            continue

        # Seek to the adjusted time (in milliseconds)
        cap.set(cv2.CAP_PROP_POS_MSEC, adjusted_sec * 1000)
        ret, frame = cap.read()
        if not ret:
            print(f"Could not read frame for row {row['index']} at {adjusted_time_str}.")
            row['frame_filename'] = ""
        else:
            instrument = row['groundtruth_toolname'].strip() or row['commercial_toolname'].strip()
            instrument_clean = instrument.replace(" ", "_")
            frame_filename = f"row{row['index']}_{instrument_clean}.png"
            frame_filepath = os.path.join(frames_output_dir, frame_filename)
            cv2.imwrite(frame_filepath, frame)
            row['frame_filename'] = frame_filename
            print(f"Saved frame for row {row['index']} at {adjusted_time_str} as {frame_filename}")

        row['adjusted_install_case_time'] = adjusted_time_str
        writer.writerow(row)

cap.release()
print(f"New CSV file written to {output_csv}")


Video FPS: 60.0
Total frames: 151133.0, Duration: 2518.88 seconds
Saved frame for row 0 at 00:07:28.796000 as row0_stapler.png
Saved frame for row 1 at 00:08:01.979000 as row1_cadiere_forceps.png
Saved frame for row 2 at 00:08:48.729000 as row2_cadiere_forceps.png
Saved frame for row 3 at 00:17:22.659000 as row3_vessel_sealer.png
Saved frame for row 4 at 00:23:07.499000 as row4_Unknown_Instrument.png
Saved frame for row 5 at 00:23:07.499000 as row5_nan(camera_in).png
Saved frame for row 6 at 00:24:58.379000 as row6_Unknown_Instrument.png
Saved frame for row 7 at 00:24:58.379000 as row7_force_bipolar.png
Saved frame for row 8 at 00:24:58.379000 as row8_force_bipolar.png
Saved frame for row 9 at 00:25:00.579000 as row9_prograsp_forceps.png
Saved frame for row 10 at 00:25:00.579000 as row10_prograsp_forceps.png
Saved frame for row 11 at 00:25:05.359000 as row11_permanent_cautery_hook/spatula.png
Saved frame for row 12 at 00:26:56.959000 as row12_prograsp_forceps.png
Saved frame for row 13